In [1]:
#!/usr/bin/env python3
"""
Normalized all-neuron poor man's classifier.

Goal
----
Use ALL cleaned neurons, without greedy selection.

This version tests whether animate/inanimate separation survives after
normalizing neural population response vectors.

Core idea
---------
Instead of using raw response magnitudes, we classify by population response
direction.

For each stimulus-level neural response vector x_i:

    x_hat_i = x_i / ||x_i||

On the training set:

    mu0 = mean normalized response for inanimate stimuli
    mu1 = mean normalized response for animate stimuli

Then normalize the class templates:

    mu0_hat = mu0 / ||mu0||
    mu1_hat = mu1 / ||mu1||

Classifier direction:

    w = mu1_hat - mu0_hat

Score:

    score_i = x_hat_i @ w

Prediction:

    animate if score_i > 0

Interpretation
--------------
This is equivalent to assigning each normalized response vector to whichever
class template has larger cosine similarity.

If score_i > 0:

    cos(x_hat_i, mu1_hat) > cos(x_hat_i, mu0_hat)

So this classifier asks whether the neural population activity pattern points
more toward the animate mean direction or the inanimate mean direction.

Expected files
--------------
/home/maria/Science/data/
    hybrid_neural_responses_reduced.npy
    google_vit-base-patch16-224_embeddings_logits.pkl

Optional:
    stimulus_ids.npy

Outputs
-------
/home/maria/Science/thesis/experiments/007--PoorMansClassifier/
    normalized_all_neurons_poor_mans_classifier/
        vit_derived_labels.csv
        stimulus_presentation_counts.csv
        kept_neuron_indices.csv
        train_test_split.csv
        all_neuron_scores.csv
        all_neuron_weight_vector.npy
        class_templates.npz
        summary.json
"""

from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    roc_auc_score,
    confusion_matrix,
)
from sklearn.model_selection import train_test_split


# =============================================================================
# Config
# =============================================================================

BASE_DIR = Path("/home/maria/Science/thesis/experiments/007--PoorMansClassifier")
DATA_DIR = Path("/home/maria/Science/data")

OUT_DIR = BASE_DIR / "normalized_all_neurons_poor_mans_classifier"
OUT_DIR.mkdir(exist_ok=True, parents=True)

NEURAL_FILE = DATA_DIR / "hybrid_neural_responses_reduced.npy"
VIT_FILE = DATA_DIR / "google_vit-base-patch16-224_embeddings_logits.pkl"
VIT_KEY = "natural_scenes"

N_STIMULI = 118
RANDOM_SEED = 42

ANIMATE_TOP1_THRESHOLD = 397

PRESENTATION_ORDER = "block"
STIMULUS_IDS_FILE = DATA_DIR / "stimulus_ids.npy"

EPS = 1e-8

# Optional:
# If True, each neuron is z-scored using train statistics before row-normalization.
# I would start with False, because this preserves the original neural response geometry.
STANDARDIZE_NEURONS_USING_TRAIN = False


# =============================================================================
# Loading
# =============================================================================

def load_neural_presentations() -> np.ndarray:
    """
    Load neural matrix and return shape:

        presentations x neurons

    Expected raw shape is usually:

        neurons x presentations = (39209, 5900)
    """
    if not NEURAL_FILE.exists():
        raise FileNotFoundError(f"Missing neural file: {NEURAL_FILE}")

    X_raw = np.asarray(np.load(NEURAL_FILE, allow_pickle=True))

    print(f"[INFO] Raw neural shape: {X_raw.shape}")

    if X_raw.ndim != 2:
        raise ValueError(f"Expected 2D neural matrix, got {X_raw.shape}")

    n0, n1 = X_raw.shape

    if n0 > n1 and n1 % N_STIMULI == 0:
        print("[INFO] Interpreting raw neural matrix as neurons x presentations.")
        X_pres = X_raw.T
    elif n1 > n0 and n0 % N_STIMULI == 0:
        print("[INFO] Interpreting raw neural matrix as presentations x neurons.")
        X_pres = X_raw
    else:
        raise ValueError(
            f"Could not infer orientation from neural shape {X_raw.shape}. "
            "Expected something like (39209, 5900) or (5900, 39209)."
        )

    X_pres = X_pres.astype(np.float32, copy=False)

    print(f"[INFO] Presentation-level neural shape: {X_pres.shape}")

    return X_pres


def load_vit_natural_scenes_logits() -> np.ndarray:
    """
    Load ViT logits for natural scenes.

    Expected output shape:

        (118, 1000)
    """
    if not VIT_FILE.exists():
        raise FileNotFoundError(f"Missing ViT file: {VIT_FILE}")

    obj = np.load(VIT_FILE, allow_pickle=True)

    if hasattr(obj, "keys"):
        keys = list(obj.keys())

        if VIT_KEY not in keys:
            raise KeyError(
                f"Key {VIT_KEY!r} not found in {VIT_FILE}. "
                f"Available keys: {keys}"
            )

        logits = np.asarray(obj[VIT_KEY])

    elif isinstance(obj, np.ndarray) and obj.dtype == object:
        item = obj.item()

        if not isinstance(item, dict):
            raise TypeError(f"Expected object array containing dict, got {type(item)}")

        if VIT_KEY not in item:
            raise KeyError(
                f"Key {VIT_KEY!r} not found in object dict. "
                f"Available keys: {list(item.keys())}"
            )

        logits = np.asarray(item[VIT_KEY])

    else:
        raise TypeError(f"Unsupported ViT object type: {type(obj)}")

    if logits.ndim != 2:
        raise ValueError(f"Expected 2D ViT logits, got {logits.shape}")

    if logits.shape[0] != N_STIMULI:
        raise ValueError(
            f"Expected {N_STIMULI} rows, got {logits.shape[0]}"
        )

    print(f"[INFO] ViT logits shape: {logits.shape}")

    return logits.astype(np.float32, copy=False)


def make_labels_from_vit_logits(logits: np.ndarray) -> np.ndarray:
    """
    Derive binary labels.

        1 = animate
        0 = inanimate

    Rule:

        top1 <= 397 => animate
    """
    top1 = np.argmax(logits, axis=1)
    y = (top1 <= ANIMATE_TOP1_THRESHOLD).astype(int)

    print("[INFO] Derived animate/inanimate labels from ViT top-1.")
    print(f"[INFO] Inanimate count: {int((y == 0).sum())}")
    print(f"[INFO] Animate count:   {int((y == 1).sum())}")

    pd.DataFrame(
        {
            "stimulus_index": np.arange(N_STIMULI),
            "top1_imagenet_class": top1,
            "label_animate": y,
        }
    ).to_csv(OUT_DIR / "vit_derived_labels.csv", index=False)

    return y


# =============================================================================
# Presentation averaging
# =============================================================================

def make_presentation_stimulus_ids(n_presentations: int) -> np.ndarray:
    """
    Return vector of length n_presentations containing stimulus IDs 0..117.
    """
    if STIMULUS_IDS_FILE.exists():
        stim_ids = np.load(STIMULUS_IDS_FILE, allow_pickle=True).astype(int).ravel()

        if len(stim_ids) != n_presentations:
            raise ValueError(
                f"{STIMULUS_IDS_FILE} has length {len(stim_ids)}, "
                f"but neural data has {n_presentations} presentations."
            )

        if stim_ids.min() < 0 or stim_ids.max() >= N_STIMULI:
            raise ValueError(
                f"Stimulus IDs must be in [0, {N_STIMULI - 1}], "
                f"got min={stim_ids.min()}, max={stim_ids.max()}."
            )

        print(f"[INFO] Loaded explicit stimulus IDs from {STIMULUS_IDS_FILE}")
        return stim_ids

    if n_presentations % N_STIMULI != 0:
        raise ValueError(
            f"n_presentations={n_presentations} is not divisible by {N_STIMULI}."
        )

    repeats = n_presentations // N_STIMULI

    if PRESENTATION_ORDER == "block":
        stim_ids = np.repeat(np.arange(N_STIMULI), repeats)
    elif PRESENTATION_ORDER == "cycle":
        stim_ids = np.tile(np.arange(N_STIMULI), repeats)
    else:
        raise ValueError("PRESENTATION_ORDER must be either 'block' or 'cycle'.")

    print(
        f"[WARN] No explicit {STIMULUS_IDS_FILE.name} found. "
        f"Assuming PRESENTATION_ORDER={PRESENTATION_ORDER!r}. "
        f"Repeats per stimulus={repeats}."
    )

    return stim_ids.astype(int)


def average_presentations_by_stimulus(
    X_pres: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Average presentation-level neural matrix to stimulus-level matrix.

    Input:

        X_pres: presentations x neurons

    Output:

        X_avg: stimuli x neurons
        counts: presentations per stimulus
    """
    n_presentations, n_neurons = X_pres.shape
    stim_ids = make_presentation_stimulus_ids(n_presentations)

    X_avg = np.zeros((N_STIMULI, n_neurons), dtype=np.float32)
    counts = np.zeros(N_STIMULI, dtype=int)

    for stim_id in range(N_STIMULI):
        mask = stim_ids == stim_id
        counts[stim_id] = int(mask.sum())

        if counts[stim_id] == 0:
            raise ValueError(f"Stimulus {stim_id} has zero presentations.")

        X_avg[stim_id] = X_pres[mask].mean(axis=0)

    print("[INFO] Averaged neural responses by stimulus.")
    print(f"[INFO] Stimulus-averaged neural shape: {X_avg.shape}")
    print(f"[INFO] Presentations per stimulus: min={counts.min()}, max={counts.max()}")

    pd.DataFrame(
        {
            "stimulus_index": np.arange(N_STIMULI),
            "n_presentations": counts,
        }
    ).to_csv(OUT_DIR / "stimulus_presentation_counts.csv", index=False)

    return X_avg, counts


# =============================================================================
# Cleaning and splitting
# =============================================================================

def make_stratified_train_test_split(
    X: np.ndarray,
    y: np.ndarray,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Make 80/20 stratified split over the 118 stimulus-level rows.
    """
    indices = np.arange(len(y))

    train_idx, test_idx = train_test_split(
        indices,
        test_size=0.2,
        stratify=y,
        random_state=RANDOM_SEED,
        shuffle=True,
    )

    split_name = np.full(len(y), "train", dtype=object)
    split_name[test_idx] = "test"

    pd.DataFrame(
        {
            "stimulus_index": np.arange(len(y)),
            "label_animate": y,
            "split": split_name,
        }
    ).to_csv(OUT_DIR / "train_test_split.csv", index=False)

    print("[INFO] Stratified 80/20 train/test split:")
    for name, idx in [("train", train_idx), ("test", test_idx)]:
        counts = np.bincount(y[idx], minlength=2)
        print(
            f"  {name:5s}: n={len(idx):3d}, "
            f"inanimate={counts[0]:3d}, animate={counts[1]:3d}"
        )

    return X[train_idx], X[test_idx], y[train_idx], y[test_idx], train_idx, test_idx


def clean_features_using_train(
    X_train: np.ndarray,
    X_test: np.ndarray,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Remove neurons that are non-finite anywhere or have zero variance on train.

    Zero-variance is checked on train because the model only learns from train.
    """
    finite = np.isfinite(X_train).all(axis=0) & np.isfinite(X_test).all(axis=0)
    train_var = np.nanvar(X_train, axis=0)
    nonzero_train_var = train_var > 0

    keep = finite & nonzero_train_var
    kept_original_indices = np.where(keep)[0]

    removed = X_train.shape[1] - int(keep.sum())
    if removed:
        print(f"[WARN] Removing {removed} non-finite or train-zero-variance neurons.")

    X_train_clean = X_train[:, keep].astype(np.float32, copy=False)
    X_test_clean = X_test[:, keep].astype(np.float32, copy=False)

    pd.DataFrame(
        {
            "clean_feature_index": np.arange(len(kept_original_indices)),
            "original_neuron_index": kept_original_indices,
        }
    ).to_csv(OUT_DIR / "kept_neuron_indices.csv", index=False)

    print(f"[INFO] Clean train shape: {X_train_clean.shape}")
    print(f"[INFO] Clean test shape:  {X_test_clean.shape}")

    return X_train_clean, X_test_clean, kept_original_indices


def standardize_neurons_using_train(
    X_train: np.ndarray,
    X_test: np.ndarray,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Optional neuron-wise z-scoring using train statistics only.

    This is OFF by default because it changes the metric of neural space.
    """
    mean = X_train.mean(axis=0)
    std = X_train.std(axis=0)

    std_safe = np.where(std > EPS, std, 1.0)

    X_train_z = (X_train - mean) / std_safe
    X_test_z = (X_test - mean) / std_safe

    return (
        X_train_z.astype(np.float32, copy=False),
        X_test_z.astype(np.float32, copy=False),
        mean.astype(np.float32, copy=False),
        std_safe.astype(np.float32, copy=False),
    )


# =============================================================================
# Normalization and classifier
# =============================================================================

def l2_normalize_rows(
    X: np.ndarray,
    eps: float = EPS,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Normalize each sample/stimulus vector to unit L2 norm.

    Returns:
        X_hat: row-normalized matrix
        norms: original row norms
    """
    norms = np.linalg.norm(X, axis=1, keepdims=True)
    norms_safe = np.maximum(norms, eps)

    X_hat = X / norms_safe

    return X_hat.astype(np.float32, copy=False), norms.ravel().astype(np.float32)


def l2_normalize_vector(
    v: np.ndarray,
    eps: float = EPS,
) -> tuple[np.ndarray, float]:
    norm = float(np.linalg.norm(v))
    norm_safe = max(norm, eps)

    return (v / norm_safe).astype(np.float32, copy=False), norm


def fit_normalized_poor_mans_classifier(
    X_train_hat: np.ndarray,
    y_train: np.ndarray,
) -> dict[str, np.ndarray | float]:
    """
    Fit normalized template classifier.

    X_train_hat should already have unit-norm rows.

    Class templates are class means of normalized response vectors,
    then each template is itself normalized to unit length.

    Score:

        score = x_hat @ (mu1_hat - mu0_hat)

    This is equivalent to comparing cosine similarity to the two templates.
    """
    if set(np.unique(y_train)) != {0, 1}:
        raise ValueError("Expected both labels 0 and 1 in y_train.")

    mu0 = X_train_hat[y_train == 0].mean(axis=0)
    mu1 = X_train_hat[y_train == 1].mean(axis=0)

    mu0_hat, mu0_norm = l2_normalize_vector(mu0)
    mu1_hat, mu1_norm = l2_normalize_vector(mu1)

    w = (mu1_hat - mu0_hat).astype(np.float32, copy=False)

    # Template cosine similarity. Since both templates are unit vectors:
    # score > 0 iff cos_to_animate > cos_to_inanimate.
    return {
        "mu0": mu0.astype(np.float32, copy=False),
        "mu1": mu1.astype(np.float32, copy=False),
        "mu0_hat": mu0_hat,
        "mu1_hat": mu1_hat,
        "mu0_norm": float(mu0_norm),
        "mu1_norm": float(mu1_norm),
        "w": w,
        "w_norm": float(np.linalg.norm(w)),
        "template_cosine_similarity": float(np.dot(mu0_hat, mu1_hat)),
    }


def compute_scores(
    X_hat: np.ndarray,
    clf: dict[str, np.ndarray | float],
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Return:
        score = cos_to_animate - cos_to_inanimate
        cos_to_inanimate
        cos_to_animate
    """
    mu0_hat = clf["mu0_hat"]
    mu1_hat = clf["mu1_hat"]
    w = clf["w"]

    assert isinstance(mu0_hat, np.ndarray)
    assert isinstance(mu1_hat, np.ndarray)
    assert isinstance(w, np.ndarray)

    cos0 = X_hat @ mu0_hat
    cos1 = X_hat @ mu1_hat
    scores = X_hat @ w

    return (
        scores.astype(np.float32, copy=False),
        cos0.astype(np.float32, copy=False),
        cos1.astype(np.float32, copy=False),
    )


def scores_to_predictions(scores: np.ndarray) -> np.ndarray:
    return (scores > 0).astype(int)


def safe_auc(y_true: np.ndarray, scores: np.ndarray) -> float:
    if len(np.unique(y_true)) < 2:
        return float("nan")
    return float(roc_auc_score(y_true, scores))


def metric_dict(y_true: np.ndarray, scores: np.ndarray) -> dict[str, object]:
    preds = scores_to_predictions(scores)
    cm = confusion_matrix(y_true, preds, labels=[0, 1])

    return {
        "accuracy": float(accuracy_score(y_true, preds)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, preds)),
        "auc": safe_auc(y_true, scores),
        "confusion_matrix_labels": ["inanimate_0", "animate_1"],
        "confusion_matrix": cm.tolist(),
    }


# =============================================================================
# Main
# =============================================================================

def main() -> None:
    np.random.seed(RANDOM_SEED)

    print("=" * 80)
    print("Loading neural data")
    print("=" * 80)

    X_pres = load_neural_presentations()

    print("=" * 80)
    print("Averaging presentations by stimulus")
    print("=" * 80)

    X_avg, presentation_counts = average_presentations_by_stimulus(X_pres)

    print("=" * 80)
    print("Loading ViT logits and labels")
    print("=" * 80)

    vit_logits = load_vit_natural_scenes_logits()
    y = make_labels_from_vit_logits(vit_logits)

    if X_avg.shape[0] != len(y):
        raise ValueError(
            f"X has {X_avg.shape[0]} rows, but y has {len(y)} labels."
        )

    print("=" * 80)
    print("Making stratified 80/20 train/test split")
    print("=" * 80)

    X_train_raw, X_test_raw, y_train, y_test, train_idx, test_idx = (
        make_stratified_train_test_split(X_avg, y)
    )

    print("=" * 80)
    print("Cleaning features using train statistics")
    print("=" * 80)

    X_train, X_test, kept_original_neuron_indices = clean_features_using_train(
        X_train_raw,
        X_test_raw,
    )

    if STANDARDIZE_NEURONS_USING_TRAIN:
        print("=" * 80)
        print("Standardizing neurons using train statistics")
        print("=" * 80)

        X_train, X_test, neuron_mean, neuron_std = standardize_neurons_using_train(
            X_train,
            X_test,
        )

        np.savez_compressed(
            OUT_DIR / "neuron_standardization_stats.npz",
            neuron_mean=neuron_mean,
            neuron_std=neuron_std,
            kept_original_neuron_indices=kept_original_neuron_indices,
        )

    print("=" * 80)
    print("L2-normalizing population response vectors")
    print("=" * 80)

    X_train_hat, train_row_norms = l2_normalize_rows(X_train)
    X_test_hat, test_row_norms = l2_normalize_rows(X_test)

    print(
        "[INFO] Train row norms before normalization: "
        f"min={train_row_norms.min():.6g}, "
        f"median={np.median(train_row_norms):.6g}, "
        f"max={train_row_norms.max():.6g}"
    )
    print(
        "[INFO] Test row norms before normalization: "
        f"min={test_row_norms.min():.6g}, "
        f"median={np.median(test_row_norms):.6g}, "
        f"max={test_row_norms.max():.6g}"
    )

    print("=" * 80)
    print("Fitting normalized all-neuron poor man's classifier")
    print("=" * 80)

    clf = fit_normalized_poor_mans_classifier(X_train_hat, y_train)

    train_scores, train_cos0, train_cos1 = compute_scores(X_train_hat, clf)
    test_scores, test_cos0, test_cos1 = compute_scores(X_test_hat, clf)

    train_metrics = metric_dict(y_train, train_scores)
    test_metrics = metric_dict(y_test, test_scores)

    print("[RESULT] Train metrics:")
    print(json.dumps(train_metrics, indent=2))

    print("[RESULT] Test metrics:")
    print(json.dumps(test_metrics, indent=2))

    print(
        "[INFO] Class template cosine similarity "
        f"cos(mu0_hat, mu1_hat) = {clf['template_cosine_similarity']:.6f}"
    )
    print(f"[INFO] ||w|| = {clf['w_norm']:.6f}")

    # Save model artifacts.
    np.save(OUT_DIR / "all_neuron_weight_vector.npy", clf["w"])

    np.savez_compressed(
        OUT_DIR / "class_templates.npz",
        mu0=clf["mu0"],
        mu1=clf["mu1"],
        mu0_hat=clf["mu0_hat"],
        mu1_hat=clf["mu1_hat"],
        w=clf["w"],
        kept_original_neuron_indices=kept_original_neuron_indices,
    )

    # Save per-stimulus scores.
    score_rows = []

    for local_pos, stim_idx in enumerate(train_idx):
        score_rows.append(
            {
                "stimulus_index": int(stim_idx),
                "split": "train",
                "label_animate": int(y[stim_idx]),
                "score": float(train_scores[local_pos]),
                "prediction_animate": int(train_scores[local_pos] > 0),
                "cos_to_inanimate_template": float(train_cos0[local_pos]),
                "cos_to_animate_template": float(train_cos1[local_pos]),
                "original_population_norm": float(train_row_norms[local_pos]),
            }
        )

    for local_pos, stim_idx in enumerate(test_idx):
        score_rows.append(
            {
                "stimulus_index": int(stim_idx),
                "split": "test",
                "label_animate": int(y[stim_idx]),
                "score": float(test_scores[local_pos]),
                "prediction_animate": int(test_scores[local_pos] > 0),
                "cos_to_inanimate_template": float(test_cos0[local_pos]),
                "cos_to_animate_template": float(test_cos1[local_pos]),
                "original_population_norm": float(test_row_norms[local_pos]),
            }
        )

    scores_df = pd.DataFrame(score_rows).sort_values("stimulus_index")
    scores_df.to_csv(OUT_DIR / "all_neuron_scores.csv", index=False)

    summary = {
        "experiment": "normalized_all_neuron_poor_mans_classifier",
        "description": (
            "Uses all cleaned neurons. Each stimulus-level population vector is "
            "L2-normalized. Class templates are means of normalized train vectors, "
            "then class templates are L2-normalized. Prediction is based on cosine "
            "similarity to animate vs inanimate templates."
        ),
        "neural_file": str(NEURAL_FILE),
        "vit_file": str(VIT_FILE),
        "vit_key": VIT_KEY,
        "n_stimuli": int(N_STIMULI),
        "presentation_level_shape": list(X_pres.shape),
        "stimulus_averaged_shape": list(X_avg.shape),
        "clean_train_shape": list(X_train.shape),
        "clean_test_shape": list(X_test.shape),
        "n_clean_neurons": int(X_train.shape[1]),
        "used_all_clean_neurons": True,
        "greedy_selection": False,
        "standardize_neurons_using_train": bool(STANDARDIZE_NEURONS_USING_TRAIN),
        "row_l2_normalization": True,
        "class_template_l2_normalization": True,
        "decision_rule": "predict animate if x_hat @ (mu1_hat - mu0_hat) > 0",
        "presentation_order_assumption": PRESENTATION_ORDER,
        "used_explicit_stimulus_ids": bool(STIMULUS_IDS_FILE.exists()),
        "min_presentations_per_stimulus": int(presentation_counts.min()),
        "max_presentations_per_stimulus": int(presentation_counts.max()),
        "class_counts_all": {
            "inanimate": int((y == 0).sum()),
            "animate": int((y == 1).sum()),
        },
        "class_counts_train": {
            "inanimate": int((y_train == 0).sum()),
            "animate": int((y_train == 1).sum()),
        },
        "class_counts_test": {
            "inanimate": int((y_test == 0).sum()),
            "animate": int((y_test == 1).sum()),
        },
        "test_fraction": 0.2,
        "train_metrics": train_metrics,
        "test_metrics": test_metrics,
        "template_mu0_norm_before_template_normalization": float(clf["mu0_norm"]),
        "template_mu1_norm_before_template_normalization": float(clf["mu1_norm"]),
        "template_cosine_similarity": float(clf["template_cosine_similarity"]),
        "weight_vector_norm": float(clf["w_norm"]),
        "train_population_norms_before_normalization": {
            "min": float(train_row_norms.min()),
            "median": float(np.median(train_row_norms)),
            "max": float(train_row_norms.max()),
        },
        "test_population_norms_before_normalization": {
            "min": float(test_row_norms.min()),
            "median": float(np.median(test_row_norms)),
            "max": float(test_row_norms.max()),
        },
    }

    with open(OUT_DIR / "summary.json", "w") as f:
        json.dump(summary, f, indent=2)

    print("=" * 80)
    print("Summary")
    print("=" * 80)
    print(json.dumps(summary, indent=2))

    print("=" * 80)
    print(f"Done. Results saved to: {OUT_DIR}")
    print("=" * 80)


if __name__ == "__main__":
    main()

Loading neural data
[INFO] Raw neural shape: (39209, 118)
[INFO] Interpreting raw neural matrix as neurons x presentations.
[INFO] Presentation-level neural shape: (118, 39209)
Averaging presentations by stimulus
[WARN] No explicit stimulus_ids.npy found. Assuming PRESENTATION_ORDER='block'. Repeats per stimulus=1.
[INFO] Averaged neural responses by stimulus.
[INFO] Stimulus-averaged neural shape: (118, 39209)
[INFO] Presentations per stimulus: min=1, max=1
Loading ViT logits and labels
[INFO] ViT logits shape: (118, 1000)
[INFO] Derived animate/inanimate labels from ViT top-1.
[INFO] Inanimate count: 55
[INFO] Animate count:   63
Making stratified 80/20 train/test split
[INFO] Stratified 80/20 train/test split:
  train: n= 94, inanimate= 44, animate= 50
  test : n= 24, inanimate= 11, animate= 13
Cleaning features using train statistics
[INFO] Clean train shape: (94, 39209)
[INFO] Clean test shape:  (24, 39209)
L2-normalizing population response vectors
[INFO] Train row norms before n

In [2]:
#!/usr/bin/env python3
"""
LOO + permutation test for normalized all-neuron poor man's classifier.

Classifier
----------
Each stimulus-level population response vector x_i is L2-normalized:

    x_hat_i = x_i / ||x_i||

For each LOO fold, fit class templates on N-1 training stimuli:

    mu0 = mean normalized response for inanimate training stimuli
    mu1 = mean normalized response for animate training stimuli

Then normalize templates:

    mu0_hat = mu0 / ||mu0||
    mu1_hat = mu1 / ||mu1||

Score held-out stimulus:

    score_i = x_hat_i @ (mu1_hat - mu0_hat)

Predict animate if score_i > 0.

Permutation test
----------------
Shuffle labels and rerun the entire LOO classifier.
This tests whether the observed LOO performance is exceptional under random labels.

Expected files
--------------
/home/maria/Science/data/
    hybrid_neural_responses_reduced.npy
    google_vit-base-patch16-224_embeddings_logits.pkl

Optional:
    stimulus_ids.npy

Outputs
-------
/home/maria/Science/thesis/experiments/007--PoorMansClassifier/
    normalized_all_neurons_loo_permutation/
        vit_derived_labels.csv
        stimulus_presentation_counts.csv
        kept_neuron_indices.csv
        loo_predictions.csv
        permutation_null.csv
        all_neuron_loo_scores.npy
        summary.json
"""

from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    roc_auc_score,
    confusion_matrix,
)


# =============================================================================
# Config
# =============================================================================

BASE_DIR = Path("/home/maria/Science/thesis/experiments/007--PoorMansClassifier")
DATA_DIR = Path("/home/maria/Science/data")

OUT_DIR = BASE_DIR / "normalized_all_neurons_loo_permutation"
OUT_DIR.mkdir(exist_ok=True, parents=True)

NEURAL_FILE = DATA_DIR / "hybrid_neural_responses_reduced.npy"
VIT_FILE = DATA_DIR / "google_vit-base-patch16-224_embeddings_logits.pkl"
VIT_KEY = "natural_scenes"

N_STIMULI = 118
ANIMATE_TOP1_THRESHOLD = 397

PRESENTATION_ORDER = "block"
STIMULUS_IDS_FILE = DATA_DIR / "stimulus_ids.npy"

RANDOM_SEED = 42
N_PERMUTATIONS = 1000

EPS = 1e-8

# Optional:
# False = preserve original neuron coordinate scale, then row-normalize.
# True = z-score each neuron across stimuli before row-normalizing.
STANDARDIZE_NEURONS = False


# =============================================================================
# Loading
# =============================================================================

def load_neural_presentations() -> np.ndarray:
    """
    Load neural matrix and return shape:

        presentations x neurons

    Expected raw shape can be:

        neurons x presentations = (39209, 118)

    or:

        presentations x neurons = (118, 39209)
    """
    if not NEURAL_FILE.exists():
        raise FileNotFoundError(f"Missing neural file: {NEURAL_FILE}")

    X_raw = np.asarray(np.load(NEURAL_FILE, allow_pickle=True))

    print(f"[INFO] Raw neural shape: {X_raw.shape}")

    if X_raw.ndim != 2:
        raise ValueError(f"Expected 2D neural matrix, got {X_raw.shape}")

    n0, n1 = X_raw.shape

    if n0 > n1 and n1 % N_STIMULI == 0:
        print("[INFO] Interpreting raw neural matrix as neurons x presentations.")
        X_pres = X_raw.T
    elif n1 > n0 and n0 % N_STIMULI == 0:
        print("[INFO] Interpreting raw neural matrix as presentations x neurons.")
        X_pres = X_raw
    else:
        raise ValueError(
            f"Could not infer orientation from neural shape {X_raw.shape}. "
            "Expected something like (39209, 118) or (118, 39209)."
        )

    X_pres = X_pres.astype(np.float32, copy=False)
    print(f"[INFO] Presentation-level neural shape: {X_pres.shape}")

    return X_pres


def load_vit_natural_scenes_logits() -> np.ndarray:
    """
    Load ViT logits for natural scenes.

    Expected output shape:

        (118, 1000)
    """
    if not VIT_FILE.exists():
        raise FileNotFoundError(f"Missing ViT file: {VIT_FILE}")

    obj = np.load(VIT_FILE, allow_pickle=True)

    if hasattr(obj, "keys"):
        keys = list(obj.keys())

        if VIT_KEY not in keys:
            raise KeyError(
                f"Key {VIT_KEY!r} not found in {VIT_FILE}. "
                f"Available keys: {keys}"
            )

        logits = np.asarray(obj[VIT_KEY])

    elif isinstance(obj, np.ndarray) and obj.dtype == object:
        item = obj.item()

        if not isinstance(item, dict):
            raise TypeError(f"Expected object array containing dict, got {type(item)}")

        if VIT_KEY not in item:
            raise KeyError(
                f"Key {VIT_KEY!r} not found in object dict. "
                f"Available keys: {list(item.keys())}"
            )

        logits = np.asarray(item[VIT_KEY])

    else:
        raise TypeError(f"Unsupported ViT object type: {type(obj)}")

    if logits.ndim != 2:
        raise ValueError(f"Expected 2D ViT logits, got {logits.shape}")

    if logits.shape[0] != N_STIMULI:
        raise ValueError(f"Expected {N_STIMULI} rows, got {logits.shape[0]}")

    print(f"[INFO] ViT logits shape: {logits.shape}")

    return logits.astype(np.float32, copy=False)


def make_labels_from_vit_logits(logits: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """
    Derive binary labels.

        1 = animate
        0 = inanimate

    Rule:

        top1 <= 397 => animate
    """
    top1 = np.argmax(logits, axis=1)
    y = (top1 <= ANIMATE_TOP1_THRESHOLD).astype(int)

    print("[INFO] Derived animate/inanimate labels from ViT top-1.")
    print(f"[INFO] Inanimate count: {int((y == 0).sum())}")
    print(f"[INFO] Animate count:   {int((y == 1).sum())}")

    pd.DataFrame(
        {
            "stimulus_index": np.arange(N_STIMULI),
            "top1_imagenet_class": top1,
            "label_animate": y,
        }
    ).to_csv(OUT_DIR / "vit_derived_labels.csv", index=False)

    return y, top1


# =============================================================================
# Presentation averaging
# =============================================================================

def make_presentation_stimulus_ids(n_presentations: int) -> np.ndarray:
    """
    Return vector of length n_presentations containing stimulus IDs 0..117.
    """
    if STIMULUS_IDS_FILE.exists():
        stim_ids = np.load(STIMULUS_IDS_FILE, allow_pickle=True).astype(int).ravel()

        if len(stim_ids) != n_presentations:
            raise ValueError(
                f"{STIMULUS_IDS_FILE} has length {len(stim_ids)}, "
                f"but neural data has {n_presentations} presentations."
            )

        if stim_ids.min() < 0 or stim_ids.max() >= N_STIMULI:
            raise ValueError(
                f"Stimulus IDs must be in [0, {N_STIMULI - 1}], "
                f"got min={stim_ids.min()}, max={stim_ids.max()}."
            )

        print(f"[INFO] Loaded explicit stimulus IDs from {STIMULUS_IDS_FILE}")
        return stim_ids

    if n_presentations % N_STIMULI != 0:
        raise ValueError(
            f"n_presentations={n_presentations} is not divisible by {N_STIMULI}."
        )

    repeats = n_presentations // N_STIMULI

    if PRESENTATION_ORDER == "block":
        stim_ids = np.repeat(np.arange(N_STIMULI), repeats)
    elif PRESENTATION_ORDER == "cycle":
        stim_ids = np.tile(np.arange(N_STIMULI), repeats)
    else:
        raise ValueError("PRESENTATION_ORDER must be either 'block' or 'cycle'.")

    print(
        f"[WARN] No explicit {STIMULUS_IDS_FILE.name} found. "
        f"Assuming PRESENTATION_ORDER={PRESENTATION_ORDER!r}. "
        f"Repeats per stimulus={repeats}."
    )

    return stim_ids.astype(int)


def average_presentations_by_stimulus(
    X_pres: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Average presentation-level neural matrix to stimulus-level matrix.

    Input:

        X_pres: presentations x neurons

    Output:

        X_avg: stimuli x neurons
        counts: presentations per stimulus
    """
    n_presentations, n_neurons = X_pres.shape
    stim_ids = make_presentation_stimulus_ids(n_presentations)

    X_avg = np.zeros((N_STIMULI, n_neurons), dtype=np.float32)
    counts = np.zeros(N_STIMULI, dtype=int)

    for stim_id in range(N_STIMULI):
        mask = stim_ids == stim_id
        counts[stim_id] = int(mask.sum())

        if counts[stim_id] == 0:
            raise ValueError(f"Stimulus {stim_id} has zero presentations.")

        X_avg[stim_id] = X_pres[mask].mean(axis=0)

    print("[INFO] Averaged neural responses by stimulus.")
    print(f"[INFO] Stimulus-averaged neural shape: {X_avg.shape}")
    print(f"[INFO] Presentations per stimulus: min={counts.min()}, max={counts.max()}")

    pd.DataFrame(
        {
            "stimulus_index": np.arange(N_STIMULI),
            "n_presentations": counts,
        }
    ).to_csv(OUT_DIR / "stimulus_presentation_counts.csv", index=False)

    return X_avg, counts


# =============================================================================
# Cleaning and normalization
# =============================================================================

def clean_features_all_data(X: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """
    Remove neurons that are non-finite anywhere or have zero variance across stimuli.

    For LOO, this is a simple unsupervised cleaning step.
    It does not use labels.
    """
    finite = np.isfinite(X).all(axis=0)
    var = np.nanvar(X, axis=0)
    nonzero_var = var > 0

    keep = finite & nonzero_var
    kept_original_indices = np.where(keep)[0]

    removed = X.shape[1] - int(keep.sum())
    if removed:
        print(f"[WARN] Removing {removed} non-finite or zero-variance neurons.")

    X_clean = X[:, keep].astype(np.float32, copy=False)

    pd.DataFrame(
        {
            "clean_feature_index": np.arange(len(kept_original_indices)),
            "original_neuron_index": kept_original_indices,
        }
    ).to_csv(OUT_DIR / "kept_neuron_indices.csv", index=False)

    print(f"[INFO] Clean stimulus-level neural shape: {X_clean.shape}")

    return X_clean, kept_original_indices


def standardize_neurons_all_data(X: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Optional neuron-wise z-scoring across stimuli.

    OFF by default. This changes the neural-space metric.
    """
    mean = X.mean(axis=0)
    std = X.std(axis=0)
    std_safe = np.where(std > EPS, std, 1.0)

    X_z = ((X - mean) / std_safe).astype(np.float32, copy=False)

    np.savez_compressed(
        OUT_DIR / "neuron_standardization_stats.npz",
        neuron_mean=mean.astype(np.float32),
        neuron_std=std_safe.astype(np.float32),
    )

    return X_z, mean.astype(np.float32), std_safe.astype(np.float32)


def l2_normalize_rows(X: np.ndarray, eps: float = EPS) -> tuple[np.ndarray, np.ndarray]:
    """
    Normalize each stimulus/population vector to unit L2 norm.
    """
    norms = np.linalg.norm(X, axis=1, keepdims=True)
    norms_safe = np.maximum(norms, eps)

    X_hat = X / norms_safe

    return X_hat.astype(np.float32, copy=False), norms.ravel().astype(np.float32)


def l2_normalize_vector(v: np.ndarray, eps: float = EPS) -> tuple[np.ndarray, float]:
    norm = float(np.linalg.norm(v))
    norm_safe = max(norm, eps)
    return (v / norm_safe).astype(np.float32, copy=False), norm


# =============================================================================
# LOO classifier
# =============================================================================

def fit_templates(
    X_train_hat: np.ndarray,
    y_train: np.ndarray,
) -> dict[str, np.ndarray | float]:
    """
    Fit normalized class templates on training data.
    """
    if np.sum(y_train == 0) == 0 or np.sum(y_train == 1) == 0:
        raise ValueError("LOO training fold must contain both classes.")

    mu0 = X_train_hat[y_train == 0].mean(axis=0)
    mu1 = X_train_hat[y_train == 1].mean(axis=0)

    mu0_hat, mu0_norm = l2_normalize_vector(mu0)
    mu1_hat, mu1_norm = l2_normalize_vector(mu1)

    w = (mu1_hat - mu0_hat).astype(np.float32, copy=False)

    return {
        "mu0_hat": mu0_hat,
        "mu1_hat": mu1_hat,
        "w": w,
        "mu0_norm": float(mu0_norm),
        "mu1_norm": float(mu1_norm),
        "template_cosine_similarity": float(np.dot(mu0_hat, mu1_hat)),
        "w_norm": float(np.linalg.norm(w)),
    }


def loo_scores_for_labels(
    X_hat: np.ndarray,
    y: np.ndarray,
) -> dict[str, np.ndarray]:
    """
    Run leave-one-out classification.

    Returns arrays of:
        scores
        predictions
        cos_to_inanimate_template
        cos_to_animate_template
        template_cosines
        weight_norms
    """
    n = X_hat.shape[0]

    scores = np.zeros(n, dtype=np.float32)
    preds = np.zeros(n, dtype=int)
    cos0 = np.zeros(n, dtype=np.float32)
    cos1 = np.zeros(n, dtype=np.float32)
    template_cosines = np.zeros(n, dtype=np.float32)
    weight_norms = np.zeros(n, dtype=np.float32)

    all_idx = np.arange(n)

    for held_out_idx in range(n):
        train_mask = all_idx != held_out_idx

        X_train_hat = X_hat[train_mask]
        y_train = y[train_mask]

        clf = fit_templates(X_train_hat, y_train)

        x = X_hat[held_out_idx]

        mu0_hat = clf["mu0_hat"]
        mu1_hat = clf["mu1_hat"]
        w = clf["w"]

        assert isinstance(mu0_hat, np.ndarray)
        assert isinstance(mu1_hat, np.ndarray)
        assert isinstance(w, np.ndarray)

        c0 = float(x @ mu0_hat)
        c1 = float(x @ mu1_hat)
        score = float(x @ w)

        scores[held_out_idx] = score
        preds[held_out_idx] = int(score > 0)
        cos0[held_out_idx] = c0
        cos1[held_out_idx] = c1
        template_cosines[held_out_idx] = float(clf["template_cosine_similarity"])
        weight_norms[held_out_idx] = float(clf["w_norm"])

    return {
        "scores": scores,
        "predictions": preds,
        "cos_to_inanimate_template": cos0,
        "cos_to_animate_template": cos1,
        "template_cosines": template_cosines,
        "weight_norms": weight_norms,
    }


def safe_auc(y_true: np.ndarray, scores: np.ndarray) -> float:
    if len(np.unique(y_true)) < 2:
        return float("nan")
    return float(roc_auc_score(y_true, scores))


def metric_dict(y_true: np.ndarray, scores: np.ndarray, preds: np.ndarray) -> dict[str, object]:
    cm = confusion_matrix(y_true, preds, labels=[0, 1])

    return {
        "accuracy": float(accuracy_score(y_true, preds)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, preds)),
        "auc": safe_auc(y_true, scores),
        "confusion_matrix_labels": ["inanimate_0", "animate_1"],
        "confusion_matrix": cm.tolist(),
    }


# =============================================================================
# Permutation test
# =============================================================================

def permutation_test_loo(
    X_hat: np.ndarray,
    y: np.ndarray,
    n_permutations: int,
    random_seed: int,
) -> pd.DataFrame:
    """
    Shuffle labels and rerun full LOO procedure.
    """
    rng = np.random.default_rng(random_seed)

    rows = []

    true_counts = np.bincount(y, minlength=2)
    print(
        f"[INFO] Permutation preserves global class counts by shuffling labels: "
        f"inanimate={true_counts[0]}, animate={true_counts[1]}"
    )

    for perm_idx in range(1, n_permutations + 1):
        y_perm = rng.permutation(y)

        result = loo_scores_for_labels(X_hat, y_perm)
        scores = result["scores"]
        preds = result["predictions"]

        metrics = metric_dict(y_perm, scores, preds)

        rows.append(
            {
                "perm_idx": perm_idx,
                "accuracy": metrics["accuracy"],
                "balanced_accuracy": metrics["balanced_accuracy"],
                "auc": metrics["auc"],
            }
        )

        if perm_idx == 1 or perm_idx % 25 == 0 or perm_idx == n_permutations:
            print(
                f"[PERM] {perm_idx:5d}/{n_permutations} "
                f"bal_acc={metrics['balanced_accuracy']:.4f} "
                f"auc={metrics['auc']:.4f}"
            )

    return pd.DataFrame(rows)


def permutation_p_value(
    observed: float,
    null_values: np.ndarray,
    greater_equal: bool = True,
) -> float:
    """
    Add-one smoothed permutation p-value.
    """
    if greater_equal:
        count = int(np.sum(null_values >= observed))
    else:
        count = int(np.sum(null_values <= observed))

    return float((count + 1) / (len(null_values) + 1))


# =============================================================================
# Main
# =============================================================================

def main() -> None:
    np.random.seed(RANDOM_SEED)

    print("=" * 80)
    print("Loading neural data")
    print("=" * 80)

    X_pres = load_neural_presentations()

    print("=" * 80)
    print("Averaging presentations by stimulus")
    print("=" * 80)

    X_avg, presentation_counts = average_presentations_by_stimulus(X_pres)

    print("=" * 80)
    print("Loading ViT logits and labels")
    print("=" * 80)

    vit_logits = load_vit_natural_scenes_logits()
    y, top1 = make_labels_from_vit_logits(vit_logits)

    if X_avg.shape[0] != len(y):
        raise ValueError(f"X has {X_avg.shape[0]} rows, but y has {len(y)} labels.")

    print("=" * 80)
    print("Cleaning features")
    print("=" * 80)

    X_clean, kept_original_neuron_indices = clean_features_all_data(X_avg)

    if STANDARDIZE_NEURONS:
        print("=" * 80)
        print("Standardizing neurons across stimuli")
        print("=" * 80)
        X_clean, _, _ = standardize_neurons_all_data(X_clean)

    print("=" * 80)
    print("L2-normalizing population response vectors")
    print("=" * 80)

    X_hat, row_norms = l2_normalize_rows(X_clean)

    print(
        "[INFO] Row norms before normalization: "
        f"min={row_norms.min():.6g}, "
        f"median={np.median(row_norms):.6g}, "
        f"max={row_norms.max():.6g}"
    )

    print("=" * 80)
    print("Running true-label LOO")
    print("=" * 80)

    loo_result = loo_scores_for_labels(X_hat, y)

    true_scores = loo_result["scores"]
    true_preds = loo_result["predictions"]

    true_metrics = metric_dict(y, true_scores, true_preds)

    print("[RESULT] True-label LOO metrics:")
    print(json.dumps(true_metrics, indent=2))

    np.save(OUT_DIR / "all_neuron_loo_scores.npy", true_scores)

    pred_df = pd.DataFrame(
        {
            "stimulus_index": np.arange(N_STIMULI),
            "label_animate": y,
            "top1_imagenet_class": top1,
            "loo_score": true_scores,
            "loo_prediction_animate": true_preds,
            "correct": true_preds == y,
            "cos_to_inanimate_template": loo_result["cos_to_inanimate_template"],
            "cos_to_animate_template": loo_result["cos_to_animate_template"],
            "template_cosine_similarity": loo_result["template_cosines"],
            "weight_norm": loo_result["weight_norms"],
            "original_population_norm": row_norms,
        }
    )

    pred_df.to_csv(OUT_DIR / "loo_predictions.csv", index=False)

    print("=" * 80)
    print("Running permutation test")
    print("=" * 80)

    perm_df = permutation_test_loo(
        X_hat=X_hat,
        y=y,
        n_permutations=N_PERMUTATIONS,
        random_seed=RANDOM_SEED,
    )

    perm_df.to_csv(OUT_DIR / "permutation_null.csv", index=False)

    p_bal_acc = permutation_p_value(
        observed=float(true_metrics["balanced_accuracy"]),
        null_values=perm_df["balanced_accuracy"].to_numpy(),
        greater_equal=True,
    )

    p_auc = permutation_p_value(
        observed=float(true_metrics["auc"]),
        null_values=perm_df["auc"].to_numpy(),
        greater_equal=True,
    )

    p_acc = permutation_p_value(
        observed=float(true_metrics["accuracy"]),
        null_values=perm_df["accuracy"].to_numpy(),
        greater_equal=True,
    )

    summary = {
        "experiment": "normalized_all_neuron_poor_mans_classifier_loo_permutation",
        "description": (
            "Uses all cleaned neurons. Each stimulus-level population vector is "
            "L2-normalized. For each leave-one-out fold, class templates are "
            "computed from the remaining stimuli and template-normalized. "
            "Prediction is based on cosine similarity to animate vs inanimate "
            "templates. Permutation test shuffles labels and repeats the full "
            "LOO procedure."
        ),
        "neural_file": str(NEURAL_FILE),
        "vit_file": str(VIT_FILE),
        "vit_key": VIT_KEY,
        "n_stimuli": int(N_STIMULI),
        "presentation_level_shape": list(X_pres.shape),
        "stimulus_averaged_shape": list(X_avg.shape),
        "clean_shape": list(X_clean.shape),
        "n_clean_neurons": int(X_clean.shape[1]),
        "used_all_clean_neurons": True,
        "greedy_selection": False,
        "loo": True,
        "row_l2_normalization": True,
        "class_template_l2_normalization": True,
        "standardize_neurons": bool(STANDARDIZE_NEURONS),
        "decision_rule": "predict animate if x_hat @ (mu1_hat - mu0_hat) > 0",
        "presentation_order_assumption": PRESENTATION_ORDER,
        "used_explicit_stimulus_ids": bool(STIMULUS_IDS_FILE.exists()),
        "min_presentations_per_stimulus": int(presentation_counts.min()),
        "max_presentations_per_stimulus": int(presentation_counts.max()),
        "class_counts_all": {
            "inanimate": int((y == 0).sum()),
            "animate": int((y == 1).sum()),
        },
        "true_loo_metrics": true_metrics,
        "n_permutations": int(N_PERMUTATIONS),
        "permutation_p_values": {
            "accuracy": p_acc,
            "balanced_accuracy": p_bal_acc,
            "auc": p_auc,
        },
        "permutation_null_summary": {
            "accuracy_mean": float(perm_df["accuracy"].mean()),
            "accuracy_std": float(perm_df["accuracy"].std(ddof=1)),
            "accuracy_95pct": float(np.quantile(perm_df["accuracy"], 0.95)),
            "balanced_accuracy_mean": float(perm_df["balanced_accuracy"].mean()),
            "balanced_accuracy_std": float(perm_df["balanced_accuracy"].std(ddof=1)),
            "balanced_accuracy_95pct": float(np.quantile(perm_df["balanced_accuracy"], 0.95)),
            "auc_mean": float(perm_df["auc"].mean()),
            "auc_std": float(perm_df["auc"].std(ddof=1)),
            "auc_95pct": float(np.quantile(perm_df["auc"], 0.95)),
        },
        "template_cosine_similarity_across_loo": {
            "min": float(np.min(loo_result["template_cosines"])),
            "median": float(np.median(loo_result["template_cosines"])),
            "max": float(np.max(loo_result["template_cosines"])),
        },
        "weight_norm_across_loo": {
            "min": float(np.min(loo_result["weight_norms"])),
            "median": float(np.median(loo_result["weight_norms"])),
            "max": float(np.max(loo_result["weight_norms"])),
        },
        "population_norms_before_normalization": {
            "min": float(row_norms.min()),
            "median": float(np.median(row_norms)),
            "max": float(row_norms.max()),
        },
    }

    with open(OUT_DIR / "summary.json", "w") as f:
        json.dump(summary, f, indent=2)

    print("=" * 80)
    print("Permutation summary")
    print("=" * 80)
    print(
        f"Observed balanced accuracy: {true_metrics['balanced_accuracy']:.4f}\n"
        f"Null balanced accuracy:     "
        f"{summary['permutation_null_summary']['balanced_accuracy_mean']:.4f} ± "
        f"{summary['permutation_null_summary']['balanced_accuracy_std']:.4f}\n"
        f"p_bal_acc:                  {p_bal_acc:.6f}\n"
        f"\n"
        f"Observed AUC:               {true_metrics['auc']:.4f}\n"
        f"Null AUC:                   "
        f"{summary['permutation_null_summary']['auc_mean']:.4f} ± "
        f"{summary['permutation_null_summary']['auc_std']:.4f}\n"
        f"p_auc:                      {p_auc:.6f}"
    )

    print("=" * 80)
    print("Full summary")
    print("=" * 80)
    print(json.dumps(summary, indent=2))

    print("=" * 80)
    print(f"Done. Results saved to: {OUT_DIR}")
    print("=" * 80)


if __name__ == "__main__":
    main()

Loading neural data
[INFO] Raw neural shape: (39209, 118)
[INFO] Interpreting raw neural matrix as neurons x presentations.
[INFO] Presentation-level neural shape: (118, 39209)
Averaging presentations by stimulus
[WARN] No explicit stimulus_ids.npy found. Assuming PRESENTATION_ORDER='block'. Repeats per stimulus=1.
[INFO] Averaged neural responses by stimulus.
[INFO] Stimulus-averaged neural shape: (118, 39209)
[INFO] Presentations per stimulus: min=1, max=1
Loading ViT logits and labels
[INFO] ViT logits shape: (118, 1000)
[INFO] Derived animate/inanimate labels from ViT top-1.
[INFO] Inanimate count: 55
[INFO] Animate count:   63
Cleaning features
[INFO] Clean stimulus-level neural shape: (118, 39209)
L2-normalizing population response vectors
[INFO] Row norms before normalization: min=11.017, median=15.5619, max=21.9945
Running true-label LOO
[RESULT] True-label LOO metrics:
{
  "accuracy": 0.711864406779661,
  "balanced_accuracy": 0.7128427128427128,
  "auc": 0.7797979797979798,
  